# Incremental TMAP: save, load & add_points

This notebook demonstrates the full workflow for building a TMAP,
saving it to disk, and later adding new molecules without re-fitting.

**Day 1** — Fit a TMAP on a base set of molecules, explore it, save the model.  
**Day 2** — Load the saved model, add new molecules, see where they land.

In [3]:
from pathlib import Path
# --- Configuration ---
CSV_PATH = Path("cluster_65053.csv")
FP_BITS = 2048
FP_RADIUS = 2
N_BASE = 50000          # molecules for initial fit
N_NEW = 50             # molecules to add later
N_NEIGHBORS = 15
SEED = 42
MODEL_PATH = Path("tmap_model.tmap")


# Generate some fingerprints and properties for coloring

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Check RDKit availability
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdFingerprintGenerator, rdMolDescriptors
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False

def smiles_to_fingerprints(
    smiles_list: list[str],
    radius: int = 2,
    n_bits: int = 2048,
) -> tuple[np.ndarray, list[int], list]:
    """
    Convert SMILES strings to Morgan fingerprints.

    Args:
        smiles_list: List of SMILES strings
        radius: Morgan fingerprint radius (default 2 = ECFP4)
        n_bits: Number of bits in fingerprint

    Returns:
        fingerprints: numpy array of shape (n_valid, n_bits)
        valid_indices: indices of successfully parsed SMILES
        mols: list of RDKit molecule objects
    """
    if not RDKIT_AVAILABLE:
        raise ImportError("RDKit is required: pip install rdkit")

    # Use the newer MorganGenerator API
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

    fingerprints = []
    valid_indices = []
    mols = []

    for i, smi in tqdm(enumerate(smiles_list),
                       desc='Generating Fingerprints', total=len(smiles_list)):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = morgan_gen.GetFingerprintAsNumPy(mol)
            fingerprints.append(fp.astype(np.uint8))
            valid_indices.append(i)
            mols.append(mol)
    return np.array(fingerprints), valid_indices, mols


def compute_molecular_properties(mols: list) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute molecular properties for coloring."""
    mw = [Descriptors.MolWt(mol) for mol in mols] # type: ignore
    logp = [Descriptors.MolLogP(mol) for mol in mols] # type: ignore
    n_rings = [rdMolDescriptors.CalcNumRings(mol) for mol in mols]
    return np.array(mw), np.array(logp), np.array(n_rings)


In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Check RDKit availability
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdFingerprintGenerator, rdMolDescriptors
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False

def smiles_to_fingerprints(
    smiles_list: list[str],
    radius: int = 2,
    n_bits: int = 2048,
) -> tuple[np.ndarray, list[int], list]:
    """
    Convert SMILES strings to Morgan fingerprints.

    Args:
        smiles_list: List of SMILES strings
        radius: Morgan fingerprint radius (default 2 = ECFP4)
        n_bits: Number of bits in fingerprint

    Returns:
        fingerprints: numpy array of shape (n_valid, n_bits)
        valid_indices: indices of successfully parsed SMILES
        mols: list of RDKit molecule objects
    """
    if not RDKIT_AVAILABLE:
        raise ImportError("RDKit is required: pip install rdkit")

    # Use the newer MorganGenerator API
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

    fingerprints = []
    valid_indices = []
    mols = []

    for i, smi in tqdm(enumerate(smiles_list),
                       desc='Generating Fingerprints', total=len(smiles_list)):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = morgan_gen.GetFingerprintAsNumPy(mol)
            fingerprints.append(fp.astype(np.uint8))
            valid_indices.append(i)
            mols.append(mol)
    return np.array(fingerprints), valid_indices, mols


def compute_molecular_properties(mols: list) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute molecular properties for coloring."""
    mw = [Descriptors.MolWt(mol) for mol in mols] # type: ignore
    logp = [Descriptors.MolLogP(mol) for mol in mols] # type: ignore
    n_rings = [rdMolDescriptors.CalcNumRings(mol) for mol in mols]
    return np.array(mw), np.array(logp), np.array(n_rings)


## Load & prepare data

In [10]:
import pandas as pd
import numpy as np
df = pd.read_csv(CSV_PATH)
print(f"Total SMILES in CSV: {len(df)}")

# Shuffle and split into base (day 1) and new (day 2)
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(df))
smiles_base = df["smiles"].iloc[idx[:N_BASE]].tolist()
smiles_new = df["smiles"].iloc[idx[N_BASE : N_BASE + N_NEW]].tolist()

fps_base, valid_fps_base, mols_base= smiles_to_fingerprints(smiles_base)
fps_new, valid_fps_new, mols_new = smiles_to_fingerprints(smiles_new)
mw, logp, n_rings= compute_molecular_properties(mols_base)
mw_new, logp_new, n_rings_new= compute_molecular_properties(mols_new)
print(f"Base molecules:  {len(fps_base)}")
print(f"New molecules:   {len(fps_new)}")


Total SMILES in CSV: 64098


Generating Fingerprints: 100%|██████████| 50/50 [00:00<00:00, 1272.61it/s]


Base molecules:  50000
New molecules:   50


## Fit a first TMAP (50k molecules)

In [11]:
from tmap import TMAP
model = TMAP(
    n_neighbors=N_NEIGHBORS,
    seed=SEED,
).fit(fps_base)

print(f"Embedding: {model.embedding_.shape}")
print(f"Tree:      {model.tree_.n_nodes} nodes, {len(model.tree_.edges)} edges")


Embedding: (50000, 2)
Tree:      50000 nodes, 49999 edges


In [ ]:
logp


array([2.2881 , 3.0975 , 2.46422, ..., 3.1612 , 2.1682 , 2.9541 ],
      shape=(50000,))

In [ ]:
viz =model.to_tmapviz(include_edges=True)
viz.add_label("index", [i for i in range( viz.n_points)])
viz.add_color_layout("mw", mw.tolist())
viz.add_color_layout("n_rings", n_rings.tolist(), categorical=True)
viz.add_color_layout("logp", logp.tolist(), color='viridis')
viz.show()


/home/afloresep/work/TMAP/src/tmap/visualization/tmapviz.py:813: UserWarning: Edges are not supported in notebook mode yet and will be ignored.
  scatter = self.to_widget(width=width, height=height, controls=controls)


In [18]:
saved = model.save(MODEL_PATH)
size_kb = saved.stat().st_size / 1024
print(f"Model saved to {saved}  ({size_kb:.0f} KB)")


Model saved to tmap_model.tmap  (659970 KB)


---
# Load & add new molecules to an existing TMAP

We load the saved model and insert new molecules without re-running the entire pipeline.

In [25]:
loaded = TMAP.load(MODEL_PATH)
print(f"Loaded model: {loaded.embedding_.shape[0]} points")


Loaded model: 50000 points


In [ ]:
n_before = loaded.embedding_.shape[0]
# new_coords = loaded.add_points(fingerprints[50_001:50_012])

print(f"Embedding: {loaded.embedding_.shape}")
print(f"Tree:      {loaded.tree_.n_nodes} nodes, {len(loaded.tree_.edges)} edges")

new_coords = loaded.add_points(fps_new)
print(f"Added {new_coords.shape[0]} molecules")


Embedding: (50000, 2)
Tree:      50000 nodes, 49999 edges
Added 50 molecules


In [36]:
n_rings


array([4, 4, 4, ..., 3, 4, 3], shape=(50000,))

In [ ]:
viz = loaded.to_tmapviz()

viz.add_label("index", [i for i in range( viz.n_points)])
viz.add_color_layout("mw", mw.tolist() + mw_new.tolist())
viz.add_color_layout("n_rings", n_rings.tolist() + n_rings_new.tolist(), categorical=True)
viz.add_color_layout("logp", logp.tolist() + logp_new.tolist())
new_old = ["old"]*len(fps_base)+["new"]*len(fps_new)
viz.add_color_layout("new/old", new_old, categorical=True)
viz.add_smiles(df.smiles[:50_050])
viz.show()


/home/afloresep/work/TMAP/src/tmap/visualization/tmapviz.py:813: UserWarning: Edges are not supported in notebook mode yet and will be ignored.
  scatter = self.to_widget(width=width, height=height, controls=controls)


### Verify: base coordinates are unchanged

In [39]:
# The original base coordinates should be bit-for-bit identical
np.testing.assert_array_equal(
    loaded.embedding_[:n_before],
    model.embedding_,
)
print("Base coordinates are identical.")


Base coordinates are identical.


### Explore the tree across old → new boundary

In [40]:
# Pick the first newly added molecule and trace its path to node 0
new_node = n_before  # first added point
path_nodes = loaded.path(0, new_node)
dist = loaded.distance(0, new_node)

print(f"Path from node 0 to new node {new_node}: {len(path_nodes)} hops")
print(f"Tree distance: {dist:.4f}")
print(f"Path: {path_nodes[:6]}{'...' if len(path_nodes) > 6 else ''}")


Path from node 0 to new node 50000: 76 hops
Tree distance: 18.6602
Path: [0, 23304, 36842, 20600, 13843, 13362]...
